# Задача C — LLM для школьных вопросов

**Yandex ML Cup — трек ML**

**Цель:** отвечать на школьные вопросы (на русском) правильно, кратко и по делу — при жёстком лимите вычислений/времени, полностью офлайн в Docker-контейнере.

**Модель:** маленький чекпойнт **Qwen3** (hidden=1024, 28 слоёв, ~0.6 млрд параметров) через **vLLM**.

---

## Ключевое решение — маленькая модель + гибридная маршрутизация

Модель на 0.6 млрд быстрая и укладывается в бюджет, но **ненадёжна в арифметике** — маленькие LLM регулярно ошибаются в `47 * 23`. Поэтому решение маршрутизирует каждый вопрос:

```
вопрос
   │
   ├─ похоже на чистую арифметику?  ── да ─▶  точный вычислитель (дроби Fraction)
   │                                          → всегда верно, ~0 мс
   └─ нет ─▶  vLLM (Qwen3, greedy-декодирование)
```

Детерминированные задачи получают **детерминированный решатель**; всё остальное идёт в LLM. Та же философия, что и в задаче A (сначала точный решатель, обученная модель — запасной вариант).

## Часть 1 — точный арифметический путь

Перед вызовом модели `try_arithmetic` проверяет, действительно ли вопрос — это просто вычисление. Функция намеренно **консервативна**: срабатывает только когда уверена, иначе возвращает `None` и передаёт вопрос LLM.

Проверки по порядку:
1. Короткий вопрос (`< 140` символов).
2. Есть ключевое слово вычисления (`сколько`, `вычисли`, `посчитай`, …).
3. Есть оператор (`+ - * / :`).
4. Можно извлечь и корректно распарсить подстроку из чисел и операторов.

Важно: вычисление идёт через `fractions.Fraction` над **AST из белого списка** — не `eval()` — поэтому `1/3 + 1/6` даёт точное `1/2`, и произвольный код выполниться не может.

In [ ]:
import ast, re
from fractions import Fraction

def _eval_frac(node):
    """Safely evaluate a math AST over exact fractions. Whitelisted nodes only."""
    if isinstance(node, ast.Expression):
        return _eval_frac(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, int):
        return Fraction(node.value, 1)
    if isinstance(node, ast.Constant) and isinstance(node.value, float):
        return Fraction(str(node.value))
    if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
        return -_eval_frac(node.operand)
    if isinstance(node, ast.BinOp):
        a, b = _eval_frac(node.left), _eval_frac(node.right)
        if isinstance(node.op, ast.Add):  return a + b
        if isinstance(node.op, ast.Sub):  return a - b
        if isinstance(node.op, ast.Mult): return a * b
        if isinstance(node.op, ast.Div):  return a / b
    raise ValueError('unsupported expression')

def try_arithmetic(question):
    q = str(question).lower().strip()
    if len(q) > 140:
        return None
    if not any(w in q for w in ('сколько', 'вычисли', 'посчитай', 'найди значение', 'реши пример')):
        return None
    if not any(op in q for op in ('+', '-', '*', '/', ':')):
        return None
    candidates = re.findall(r'[0-9][0-9\s\+\-\*/:\.,\(\)]{1,90}[0-9\)]', q)
    if not candidates:
        return None
    expr = max(candidates, key=len).replace(',', '.').replace(':', '/')
    expr = re.sub(r'\s+', '', expr)
    if not re.fullmatch(r'[0-9\+\-\*/\.\(\)]+', expr):
        return None
    try:
        val = _eval_frac(ast.parse(expr, mode='eval'))
    except Exception:
        return None
    if val.denominator == 1:
        return f'Ответ: {val.numerator}.'
    return f'Ответ: {val.numerator}/{val.denominator}.'

for q in ['Вычисли 47 * 23', 'Сколько будет 1/3 + 1/6?', 'Кто написал Войну и мир?']:
    print(f'{q!r:45} -> {try_arithmetic(q)}')

## Часть 2 — путь LLM (vLLM)

Всё остальное идёт в модель Qwen3 через **vLLM** (быстрый батчевый инференс, PagedAttention). Ключевые решения:

| Параметр | Значение | Зачем |
|---|---|---|
| `temperature` | **0.0** | greedy — детерминированно, воспроизводимо, лучше для фактических ответов |
| `max_tokens` | 192 | ответы должны быть короткими; ограничивает время |
| `max_model_len` | 1024 | школьные вопросы короткие; меньше KV-кэш = выше пропускная способность |
| `dtype` | bfloat16 | влезает в GPU, без заметной потери качества |
| `gpu_memory_utilization` | 0.88 | заполняем KV-кэш настолько, насколько безопасно |

**Системный промпт** задаёт стиль: *отвечай верно, кратко, без лишних рассуждений.* Все неарифметические вопросы объединяются в один вызов `llm.generate` для скорости.

In [ ]:
# Chat-template prompt in Qwen's <|im_start|> format
SYSTEM_PROMPT = (
    'Ты помощник для школьных вопросов. '
    'Отвечай правильно, кратко и по делу. '
    'Если нужно решение — дай короткое понятное решение. '
    'Не пиши лишние рассуждения.'
)

def make_prompt(question):
    q = str(question).strip()
    return (
        '<|im_start|>system\n' + SYSTEM_PROMPT + '\n<|im_end|>\n'
        '<|im_start|>user\n'   + q             + '\n<|im_end|>\n'
        '<|im_start|>assistant\n'
    )

# --- vLLM inference (runs in the Docker container with a GPU) ---
# from vllm import LLM, SamplingParams
# llm = LLM(model='/workspace/weights', dtype='bfloat16',
#           gpu_memory_utilization=0.88, max_model_len=1024, trust_remote_code=True)
# sampling = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=192,
#                           stop=['<|im_end|>', '<|endoftext|>'])
# outputs = llm.generate([make_prompt(q) for q in questions], sampling)
print(make_prompt('Кто написал «Войну и мир»?'))

## Часть 3 — маршрутизация + очистка

Диспетчер сначала пробует арифметику, собирает всё остальное для одного батчевого вызова LLM, затем чистит сырой текст модели (убирает блоки `<think>`, маркеры chat-шаблона, обрезает «простыни»).

In [ ]:
def clean_answer(text):
    text = str(text or '')
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace('<|im_end|>', '').replace('<|endoftext|>', '')
    for marker in ('<|im_start|>user', '<|im_start|>system', '<|im_start|>assistant'):
        if marker in text:
            text = text.split(marker, 1)[0]
    return text.strip()

def answer_all(items, llm=None, sampling=None):
    """Route each question: exact arithmetic if possible, else batched vLLM."""
    answers = [None] * len(items)
    llm_idx, llm_prompts = [], []
    for i, item in enumerate(items):
        q = item.get('question', '')
        det = try_arithmetic(q)
        if det is not None:
            answers[i] = det                      # exact path
        else:
            llm_idx.append(i)
            llm_prompts.append(make_prompt(q))    # defer to model
    if llm_prompts and llm is not None:
        outputs = llm.generate(llm_prompts, sampling)   # one batched call
        for i, out in zip(llm_idx, outputs):
            answers[i] = clean_answer(out.outputs[0].text if out.outputs else '')
    return [a if a is not None else '' for a in answers]

## Почему такой дизайн

- **Маленькая модель, а не большая** — бюджет времени/вычислений вознаграждает быструю модель на 0.6 млрд, а не медленную большую. Разрыв на сложных случаях закрывается маршрутизацией, а не масштабом.
- **Детерминированный решатель для детерминированных задач** — арифметика это то, где маленькие LLM ошибаются *и* где точный вычислитель тривиален и верен на 100%. Берём эти баллы бесплатно.
- **Greedy-декодирование** — для фактических коротких ответов сэмплирование не даёт пользы; `temperature=0` воспроизводимо и не уводит в случайные ошибки.
- **Безопасное вычисление** — белый список AST + `Fraction`, никогда `eval()`, поэтому арифметический путь не выполняет произвольный код и остаётся точным (без ошибок округления float).
- **Надёжный ввод-вывод** — контейнер всегда пишет валидный (даже пустой) файл вывода, поэтому один плохой вход не обнуляет весь прогон.

## Итог

| Компонент | Роль |
|---|---|
| `try_arithmetic` | точный вычислитель на `Fraction` для чисто математических вопросов |
| Qwen3 0.6B + vLLM | быстрая батчевая LLM для всего остального, greedy-декодирование |
| системный промпт | заставляет отвечать коротко, верно, без «воды» |
| `clean_answer` | убирает следы рассуждений и chat-маркеры из сырого вывода |

**Одной строкой:** гибридная QA-система — детерминированные вопросы направляем в точный решатель, остальное в маленькую greedy-LLM — которая бьёт «просто возьми модель побольше» при жёстком лимите задержки/вычислений.